# Phase 1: Raw Exploratory Data Analysis

**Purpose**: Validate downloaded WESAD dataset integrity before merging

**What to check**:
- All expected subjects are present (according to the official dataset readme)
- Pickle files are not corrupted and can therefore be loaded
- Sampling rates are consistent
- All sensors are available per subject
- Data quality issues

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

# Use metadata extractor
import sys

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../")
from src.data.wesad_metadata_extractor import (
    determine_chest_sampling_rate,
    determine_wrist_sampling_rates,
    extract_wesad_metadata,
)

# Styling
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Imports successful!")

In [ ]:
# Dataset configuration
WESAD_PATH = Path("../data/raw/wesad/WESAD")
EXPECTED_SUBJECTS = [
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    13,
    14,
    15,
    16,
    17,
]  # S12 is missing as per official readme
LABEL_FREQ = 700  # Hz - from WESAD documentation

# Expected sensors
EXPECTED_CHEST_SENSORS = ["ACC", "ECG", "EMG", "EDA", "Temp", "Resp"]
EXPECTED_WRIST_SENSORS = ["ACC", "BVP", "EDA", "TEMP"]

print(f"WESAD Path: {WESAD_PATH.absolute()}")
print(f"Expected subjects: {len(EXPECTED_SUBJECTS)}")

## 1. Verify all expected subject folders exist

In [ ]:
print("DATASET SUBJECTS CHECK:\n")

# Check if WESAD directory exists
if not WESAD_PATH.exists():
    print(f"ERROR: WESAD directory not found at {WESAD_PATH}")
    print("Please download the dataset first")
else:
    print(f"WESAD directory found: {WESAD_PATH}\n")

    # Find all subject directories
    subject_dirs = sorted(
        [d for d in WESAD_PATH.iterdir() if d.is_dir() and d.name.startswith("S")]
    )
    found_subjects = [int(d.name[1:]) for d in subject_dirs]

    print(f"Found {len(subject_dirs)} subject directories:")
    print(f"  {[d.name for d in subject_dirs]}\n")

    # Check for missing subjects
    missing_subjects = set(EXPECTED_SUBJECTS) - set(found_subjects)
    extra_subjects = set(found_subjects) - set(EXPECTED_SUBJECTS)

    if missing_subjects:
        print(f" Missing subjects: {sorted(missing_subjects)}")
    else:
        print(" All expected subjects present")

    if extra_subjects:
        print(f" Extra subjects found: {sorted(extra_subjects)}")

    # Check for pickle files
    print("\nChecking pickle files:")
    valid_subjects = []
    missing_files = []

    for sid in found_subjects:
        pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"
        if pkl_path.exists():
            valid_subjects.append(sid)
            print(f" S{sid}.pkl")
        else:
            missing_files.append(sid)
            print(f" S{sid}.pkl MISSING")

    print(
        f"\n✓ Valid subjects with pickle files: {len(valid_subjects)}/{len(found_subjects)}"
    )

    # Save for later use
    VALID_SUBJECTS = valid_subjects

## 2. Check for possible corruption in pickle files:

In [ ]:
print("PICKLE FILE INTEGRITY CHECK:\n")

corrupted_files = []
loadable_subjects = []

for sid in VALID_SUBJECTS:
    pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"
    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f, encoding="latin1")

        # Basic structure check
        if "signal" in data and "label" in data:
            print(f"S{sid}: Loadable and has expected structure")
            loadable_subjects.append(sid)
        else:
            print(f" S{sid}: Missing 'signal' or 'label' key")
            corrupted_files.append(sid)

    except Exception as e:
        print(f" S{sid}: CORRUPTED - {str(e)[:50]}")
        corrupted_files.append(sid)

print("\n")
if corrupted_files:
    print(f" CORRUPTED FILES: {corrupted_files}")
    print("Dataset might need to be redownloaded.")
else:
    print(f" ALL FILES LOADABLE: {len(loadable_subjects)}/{len(VALID_SUBJECTS)}")

# Update valid subjects list
VALID_SUBJECTS = loadable_subjects

## 3. Use metadata extractor to get sampling rates and sensor info

In [ ]:
print("METADATA EXTRACTION")

# Store metadata for all subjects
metadata_db = {}

for sid in VALID_SUBJECTS:
    pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"

    try:
        # Extract metadata
        metadata = extract_wesad_metadata(str(pkl_path))
        chest_rate = determine_chest_sampling_rate(str(pkl_path))
        wrist_rates = determine_wrist_sampling_rates(str(pkl_path), chest_rate)

        metadata_db[sid] = {
            "metadata": metadata,
            "chest_rate": chest_rate,
            "wrist_rates": wrist_rates,
            "label_length": metadata["label_length"],
        }

        print(f"S{sid}: Extracted metadata with sucess")

    except Exception as e:
        print(f"S{sid}: Failed - {str(e)[:50]}")

print(f"\nExtracted metadata for {len(metadata_db)} subjects")

## 4. Check if all subjects have the same sampling rates

In [ ]:
print("SAMPLING RATE VERIFICATION")

# Collect sampling rates
chest_rates = [
    metadata_db[sid]["chest_rate"] for sid in metadata_db
]  # expectetion: [700, 700, 700,..]
wrist_rates_by_sensor = defaultdict(list)

for sid in metadata_db:
    for sensor, rate in metadata_db[sid]["wrist_rates"].items():
        wrist_rates_by_sensor[sensor].append(rate)

# Check chest rate consistency
print("\nChest Sampling Rates:")
unique_chest_values = set(chest_rates)
if len(unique_chest_values) == 1:
    print(f"  This is consistent: {list(unique_chest_values)[0]} Hz")
else:
    print(f"     INCONSISTENT!: {unique_chest_values}")
    print(f"     Per subject: {dict(zip(metadata_db.keys(), chest_rates))}")

# Check wrist rate consistency
print("\nWrist Sampling Rates:")
for sensor, rates in wrist_rates_by_sensor.items():
    unique_rates = set(rates)
    if len(unique_rates) == 1:
        print(f"  {sensor}: Consistent at {list(unique_rates)[0]} Hz")
    else:
        print(f"     {sensor}: INCONSISTENT - {unique_rates}")
        # Show which subjects have different rates
        for sid, rate in zip(metadata_db.keys(), rates):
            if rate != min(rates):  # Show outliers
                print(f"       S{sid}: {rate} Hz")